In [1]:
save_directory = "./models/qwen3-0.6b-base"

In [48]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load directly from your local folder
tokenizer = AutoTokenizer.from_pretrained(save_directory)
model = AutoModelForCausalLM.from_pretrained(
    save_directory,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model successfully loaded from local storage!")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model successfully loaded from local storage!


In [5]:
messages = [
    {"role": "user",
     "content": "capital of sri lanka?"}
]

In [6]:
# Apply the model's specific chat format
formatted_prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

In [7]:
# Tokenize, move to device, and generate
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

In [8]:
# Adjust generation parameters for better creativity and flow
outputs = model.generate(
    **inputs, 
    max_new_tokens=200,
    temperature=0.7,      # Controls randomness (higher = more creative)
    do_sample=True        # Required when using temperature
)

In [9]:
# Decode only the newly generated tokens
input_length = inputs.input_ids.shape[1]
response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)


print(response)

<think>
Okay, the user is asking if Sri Lanka's capital is capital. Let me think. Sri Lanka is a country in Southeast Asia. The capital is Colombo. So the answer should be yes, and mention Colombo. Also, make sure to check if there are any other capitals, but I think that's correct. Let me confirm the capital again. Yes, Colombo.
</think>

Yes, the capital of Sri Lanka is **Colombo**.


In [30]:
from transformers import pipeline, GenerationConfig
from langchain_core.language_models.llms import LLM
from typing import Any, List, Optional

In [32]:
gen_config = GenerationConfig(
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id # Prevents open-ended warning quirks
)

In [34]:
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [36]:
result = hf_pipeline(
    "Explain quantum computing",
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [38]:
result

[{'generated_text': 'Explain quantum computing and its relation to quantum information science.\n\n**Answer:**\nA quantum computer is a device that uses quantum mechanics to perform computation. It is based on the principles of quantum mechanics, such as superposition and entanglement, which allow it to perform complex computations. Quantum information science is a field that studies the properties of quantum information, such as quantum states and operations, and the interactions between quantum states and quantum systems. It is aimed at developing and applying quantum computing in various fields, such as cryptography and materials science. The two fields have a strong relationship, with quantum information science providing the theoretical foundation for quantum computing, and quantum computing enabling the practical implementation of quantum information science.\n\n**Answer:**\n**Quantum computing** is a field of study that uses quantum mechanics to perform computations. It is based

In [40]:
print(model.generation_config)

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "temperature": 0.6,
  "top_k": 20,
  "top_p": 0.95
}



In [43]:
text = result[0]["generated_text"]
print(text)

Explain quantum computing and its relation to quantum information science.

**Answer:**
A quantum computer is a device that uses quantum mechanics to perform computation. It is based on the principles of quantum mechanics, such as superposition and entanglement, which allow it to perform complex computations. Quantum information science is a field that studies the properties of quantum information, such as quantum states and operations, and the interactions between quantum states and quantum systems. It is aimed at developing and applying quantum computing in various fields, such as cryptography and materials science. The two fields have a strong relationship, with quantum information science providing the theoretical foundation for quantum computing, and quantum computing enabling the practical implementation of quantum information science.

**Answer:**
**Quantum computing** is a field of study that uses quantum mechanics to perform computations. It is based on the principles of quant